# Node class

In [45]:
import queue
from copy import deepcopy
class Node:
    def __init__(self, state, parent=None, action=None, g=0, f=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  
        self.f = f  
        if parent is None:
            self.depth = 0
        else:
            self.depth = parent.depth + 1

    def __hash__(self):
        if isinstance(self.state, list):
            state_tuple = tuple([tuple(row) for row in self.state])
            return hash(state_tuple)
        return hash(self.state)

    def __eq__(self, other):
        return isinstance(other, Node) and self.state == other.state

    def __gt__(self, other):
        return isinstance(other, Node) and self.f > other.f
    
    # Add lt for proper priority queue comparison
    def __lt__(self, other):
        return isinstance(other, Node) and self.f < other.f



Similary Function :


In [46]:
import csv
import math

def map_education_level(level):
    education_mapping = {
        'No Formal Education': 0,
        'High School': 12,
        'Technical Diploma': 14,
        "Bachelor's": 16,
        "Master's": 17,
        'Doctorate': 22,
        'Ingénieur': 19,
        'IngÃ©nieur': 19  # Handle encoding issue
    }
    return education_mapping.get(level.strip(), 0)

def noncompatibility(job, employee, k=2, M=22, weights=None):
    if weights is None:
        weights = {
            'skills': 0.2,
            'experience': 0.2,
            'salary': 0.15,
            'job_interest': 0.1,
            'sector': 0.1,
            'education': 0.15,
            'location': 0.1
        }
    total_weight = sum(weights.values())
    if total_weight != 1.0:
        raise ValueError("Weights must sum to 1.0")

    job_skills = [s.strip().lower() for s in job['skills'].split(',')]
    emp_skills = [s.strip().lower() for s in employee['skills'].split(',')]
    job_skills_set = set(job_skills)
    emp_skills_set = set(emp_skills)
    intersection = job_skills_set & emp_skills_set
    required_skills = len(job_skills_set)
    if required_skills == 0:
        skills_score = 0.0
    else:
        skills_score = 1.0 - (len(intersection) / required_skills)
    weighted_skills = skills_score * weights['skills']

    emp_exp = employee['experience']
    job_exp = job['experience']
    if emp_exp < job_exp:
        if job_exp == 0:
            exp_score = 1.0
        else:
            exp_score = 1.0 - 0.9 * (emp_exp / job_exp)
    elif job_exp <= emp_exp < 20:
        if job_exp == 20:
            exp_score = 0.0
        else:
            denominator = 20 - job_exp
            exp_score = 0.1 - 0.1 * (emp_exp - job_exp) / denominator if denominator != 0 else 0.0
    else:
        exp_score = 0.0
    weighted_exp = exp_score * weights['experience']

    E = employee['salary']
    O = job['salary']
    if O == 0:
        salary_score = 0.0 if E <= 0 else 1.0
    else:
        if E <= O:
            salary_score = 0.0
        elif E >= k * O:
            salary_score = 1.0
        else:
            salary_score = (E - O) / ((k - 1) * O)
    weighted_salary = salary_score * weights['salary']

    job_interest_score = 1.0 if employee['job_interest'].lower() != job['job_interest'].lower() else 0.0
    weighted_job_interest = job_interest_score * weights['job_interest']

    sector_score = 1.0 if employee['sector'].lower() != job['sector'].lower() else 0.0
    weighted_sector = sector_score * weights['sector']

    emp_edu = employee['education']
    job_edu = job['education']
    if emp_edu < job_edu:
        edu_score = 1.0
    elif job_edu <= emp_edu < M:
        denominator = M - job_edu
        edu_score = 0.1 - 0.1 * (emp_edu - job_edu) / denominator if denominator != 0 else 0.0
    else:
        edu_score = 0.0
    weighted_edu = edu_score * weights['education']

    location_score = 1.0 if employee['location'].lower() != job['location'].lower() else 0.0
    weighted_location = location_score * weights['location']

    total = (weighted_skills + weighted_exp + weighted_salary +
             weighted_job_interest + weighted_sector +
             weighted_edu + weighted_location)
    return total

def process_jobs_and_export(jobs_list, csv_filename, output_filename='noncompatibility_scores.csv'):
    job_seekers = []
    with open(csv_filename, mode='r', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)
        for row in reader:
            seeker = {
                'skills': row['skills'],
                'experience': int(row['experience']),
                'salary': float(row['salary']),
                'location': row['location'],
                'job_interest': row['job_interest'],
                'sector': row['sector'],
                'education': map_education_level(row['education_level']),
                'job_id': row['job_id']
            }
            job_seekers.append(seeker)

    jobs = []
    for job_attrs in jobs_list:
        job = {
            'skills': job_attrs[0],
            'experience': int(job_attrs[1]),
            'salary': float(job_attrs[2]),
            'location': job_attrs[3],
            'job_interest': job_attrs[4],
            'sector': job_attrs[5],
            'education': int(job_attrs[6]),
            'job_id': str(job_attrs[7])
        }
        jobs.append(job)

    score_matrix = []
    for seeker in job_seekers:
        seeker_scores = []
        for job in jobs:
            score = noncompatibility(job, seeker)
            seeker_scores.append(score)
        score_matrix.append(seeker_scores)

    job_ids = [job['job_id'] for job in jobs]
    header = ['Seeker ID'] + job_ids
    with open(output_filename, 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        for i, seeker in enumerate(job_seekers):
            row = [seeker['job_id']] + [f"{score:.4f}" for score in score_matrix[i]]
            writer.writerow(row)

    top_10_per_job = []
    for job_idx in range(len(jobs)):
        scores_with_employees = []
        for seeker_idx in range(len(job_seekers)):
            score = score_matrix[seeker_idx][job_idx]
            emp_id = job_seekers[seeker_idx]['job_id']
            scores_with_employees.append((score, emp_id))
        
        scores_with_employees.sort(key=lambda x: (x[0], x[1]))
        top_10 = [(emp_id, score) for score, emp_id in scores_with_employees[:10]]
        top_10_per_job.append(top_10)

    return top_10_per_job  # Return only the top employees list

if __name__ == "__main__":
    sample_jobs = [
        ["Python, SQL, Data Analysis", 3, 70000.0, "Blida", "Data Science", "IT", 16, "JOB001"],
        ["Social Media, SEO, Content Creation", 2, 50000.0, "Algiers", "Marketing", "Business", 16, "JOB002"],
        ["Networking, Security, Linux", 5, 90000.0, "Oran", "System Administration", "IT", 17, "JOB003"],
        ["3D Modeling, UI/UX, Photoshop", 2, 98636.0, "Relizane", "Art & Design", "Creative", 16, "JOB004"],
        ["MATLAB, SolidWorks, Project Management", 4, 96679.0, "Blida", "Engineering", "Engineering", 16, "JOB005"],
        ["Legal Research, Regulatory Compliance", 3, 97794.0, "Batna", "Legal", "Legal", 16, "JOB006"],
        ["Blueprint Reading, Safety Procedures", 8, 131492.0, "Chlef", "Construction", "Construction", 16, "JOB007"],
        ["Cold Calling, Customer Service", 14, 41818.0, "Oran", "Sales", "Business", 16, "JOB008"],
        ["Assessment, Curriculum Design", 7, 107518.0, "Sidi Bel AbbÃ¨s", "Education", "Education", 16, "JOB009"],
        ["Reservation Systems, Event Planning", 1, 130677.0, "Tlemcen", "Hospitality", "Tourism & Hospitality", 16, "JOB010"]
    ]

    top_10_employees = process_jobs_and_export(sample_jobs, 'job_seekers_with_ids.csv', 'scores.csv')
    print("Top 10 employees for each job:")
    for idx, job_top in enumerate(top_10_employees):
        print(f"\nJob {sample_jobs[idx][7]}:")
        for emp_id, score in job_top:
            print(f"  Employee ID: {emp_id}, Score: {score:.4f}")


Top 10 employees for each job:

Job JOB001:
  Employee ID: 17178, Score: 0.1812
  Employee ID: 15556, Score: 0.1968
  Employee ID: 18853, Score: 0.2712
  Employee ID: 19966, Score: 0.2761
  Employee ID: 15169, Score: 0.2784
  Employee ID: 17960, Score: 0.2836
  Employee ID: 14555, Score: 0.2843
  Employee ID: 16140, Score: 0.2855
  Employee ID: 12457, Score: 0.2860
  Employee ID: 19166, Score: 0.2862

Job JOB002:
  Employee ID: 10176, Score: 0.0695
  Employee ID: 10957, Score: 0.0919
  Employee ID: 13305, Score: 0.1010
  Employee ID: 14479, Score: 0.1306
  Employee ID: 13232, Score: 0.1364
  Employee ID: 13831, Score: 0.1476
  Employee ID: 12374, Score: 0.1500
  Employee ID: 13774, Score: 0.1504
  Employee ID: 19425, Score: 0.1510
  Employee ID: 12097, Score: 0.1540

Job JOB003:
  Employee ID: 11374, Score: 0.1250
  Employee ID: 10071, Score: 0.1650
  Employee ID: 17244, Score: 0.2067
  Employee ID: 12582, Score: 0.2093
  Employee ID: 13524, Score: 0.2120
  Employee ID: 11473, Score: 0

# make a problem definition where you make all the standard methods like expand node..

we need the following in problem formulation class:
is_goal()
initial_state
expand_node()


In [47]:
class Job_matching:
    def __init__(self, initial_state, goal_test, state_transition_model, actions):
        self.initial_state = initial_state
        self.goal_test = goal_test
        self.state_transition_model = state_transition_model
        self.actions = actions
        # Precompute job-employee scores for quick lookup
        self.job_emp_scores = {}
        for job_id, emp_list in self.state_transition_model.items():
            for emp_info in emp_list:
                emp_id = emp_info[0]
                score = emp_info[1]
                self.job_emp_scores[(job_id, emp_id)] = score

    def is_goal(self, state):
        # Check if all jobs are assigned
        assigned_jobs = {job_id for (job_id, emp_id) in state}
        all_jobs = set(self.state_transition_model.keys())
        if assigned_jobs != all_jobs:
            return False
        # Check if total score meets goal_test (for local search)
        if self.goal_test > 0:
            total_score = sum(self.job_emp_scores.get((job_id, emp_id), 0) for (job_id, emp_id) in state)
            return total_score <= self.goal_test
        return True  # Global search: all jobs assigned

    def expand_node(self, node):
        current_state = node.state
        assigned_employees = {emp_id for (job_id, emp_id) in current_state}
        assigned_jobs = {job_id for (job_id, emp_id) in current_state}
        all_jobs = set(self.state_transition_model.keys())
        remaining_jobs = [job_id for job_id in all_jobs if job_id not in assigned_jobs]
        
        if not remaining_jobs:
            return []
        
        next_job = remaining_jobs[0]  # Process jobs in transition model order
        child_nodes = []
        
        for emp_info in self.state_transition_model[next_job]:
            emp_id, emp_score = emp_info[0], emp_info[1]
            if emp_id in assigned_employees:
                continue
            
            new_state = list(current_state)
            new_state.append((next_job, emp_id))
            new_g = node.g + emp_score
            
            # Calculate heuristic for remaining jobs
            remaining_jobs_after = [job for job in remaining_jobs if job != next_job]
            sum_heuristic = 0
            assigned_in_child = assigned_employees.copy()
            assigned_in_child.add(emp_id)
            
            for job in remaining_jobs_after:
                for emp_candidate, score in self.state_transition_model[job]:
                    if emp_candidate not in assigned_in_child:
                        sum_heuristic += score
                        break
            
            new_f = new_g + sum_heuristic
            child_node = Node(new_state, node, emp_id, new_g, new_f)
            child_nodes.append(child_node)
        
        return child_nodes

#  General Search algorithms:

In [48]:
class GeneralSearch:
    def __init__(self, problem):
        self.problem = problem
        self.use_cost = False
        self.use_heuristic = True

    def set_frontier(self, search_strategy="A*"):
        frontier = queue.PriorityQueue()
        self.use_heuristic = True
        if search_strategy == "A*":
            self.use_cost = True     
        elif search_strategy == "Greedy":
            self.use_cost = False
        else:
            raise ValueError("Unsupported search strategy: " + str(search_strategy))
        return frontier

    def search(self, search_strategy="A*", max_depth=float('inf')):
        frontier = self.set_frontier(search_strategy)
        explored = set()
        initial_node = Node(self.problem.initial_state)
        frontier.put(initial_node)

        while not frontier.empty():
            node = frontier.get()
            if self.problem.is_goal(node.state):
                print("Goal reached!")
                return node
            
            # Convert list state to tuple for hashing
            state_tuple = tuple(tuple(pair) for pair in node.state)
            
            if node.depth > max_depth or state_tuple in explored:
                continue
                
            explored.add(state_tuple)
            child_nodes = self.problem.expand_node(node)
            
            for child_node in child_nodes:
                # Convert child state to tuple for comparison
                child_state_tuple = tuple(tuple(pair) for pair in child_node.state)
                if child_state_tuple not in explored:
                    frontier.put(child_node)
                    
        return None

In [49]:
def main():
    # Define 20 sample jobs with some in the same domain but varying requirements
    sample_jobs = [
        # Data Science domain (lower to higher)
        ["Python, SQL, Data Analysis", 2, 65000.0, "Blida", "Data Science", "IT", 16, "JOB001"],  # Lower experience
        ["Python, Machine Learning", 3, 70000.0, "Algiers", "Data Science", "IT", 16, "JOB002"],   # Medium
        ["Python, Big Data", 5, 90000.0, "Oran", "Data Science", "IT", 17, "JOB003"],              # Higher
        
        # Marketing domain
        ["Social Media, SEO", 1, 45000.0, "Relizane", "Marketing", "Business", 16, "JOB004"],      # Lower
        ["Content Strategy, Analytics", 2, 50000.0, "Blida", "Marketing", "Business", 16, "JOB005"], 
        ["SEO, Digital Marketing", 4, 75000.0, "Algiers", "Marketing", "Business", 16, "JOB006"],  # Higher
        
        # IT/System Admin
        ["Networking Basics", 3, 60000.0, "Oran", "System Administration", "IT", 16, "JOB007"],    # Lower
        ["Security, Linux", 5, 90000.0, "Relizane", "System Administration", "IT", 17, "JOB008"], 
        ["Cloud, Advanced Security", 7, 120000.0, "Blida", "System Administration", "IT", 18, "JOB009"],  # Higher
        
        # Engineering
        ["CAD Design", 2, 50000.0, "Algiers", "Engineering", "Engineering", 16, "JOB010"],          # Lower
        ["MATLAB, Simulation", 4, 80000.0, "Oran", "Engineering", "Engineering", 16, "JOB011"],     
        ["Project Management, Six Sigma", 6, 110000.0, "Relizane", "Engineering", "Engineering", 18, "JOB012"],  # Higher
        
        # Art & Design
        ["Graphic Design", 1, 40000.0, "Blida", "Art & Design", "Creative", 16, "JOB013"],         # Lower
        ["3D Modeling Basics", 2, 55000.0, "Algiers", "Art & Design", "Creative", 16, "JOB014"],  
        ["VR/AR Design", 5, 95000.0, "Oran", "Art & Design", "Creative", 17, "JOB015"],            # Higher
        
        # Other domains
        ["Legal Research", 2, 60000.0, "Relizane", "Legal", "Legal", 16, "JOB016"],
        ["Construction Safety", 5, 85000.0, "Blida", "Construction", "Construction", 16, "JOB017"],
        ["Sales Techniques", 1, 30000.0, "Algiers", "Sales", "Business", 16, "JOB018"],
        ["Curriculum Development", 4, 60000.0, "Oran", "Education", "Education", 17, "JOB019"],
        ["Hotel Management", 3, 45000.0, "Relizane", "Hospitality", "Tourism", 16, "JOB020"]
    ]
    
  
    try:
        # Get the top employees
        top_10_employees = process_jobs_and_export(sample_jobs, 'job_seekers_with_ids.csv', 'scores.csv')
        
        # Create state transition model
        state_transition_model = {job[7]: top_10_employees[idx] for idx, job in enumerate(sample_jobs)}
        
        # Initialize problem
        problem = Job_matching(
            initial_state=[],
            goal_test=0,
            state_transition_model=state_transition_model,
            actions=False
        )

        # Run search
        search = GeneralSearch(problem)
        solution_node = search.search(search_strategy="A*")

        # Display results
        if solution_node:
            print("\n=== Optimal Assignments ===")
            total_score = 0
            for assignment in solution_node.state:
                job_id = assignment[0]
                emp_id = assignment[1]
                score = problem.job_emp_scores[(job_id, emp_id)]
                total_score += score
                
                print(f"\nJob: {job_id}")
                print(f"  Employee: {emp_id}")
                print(f"  Match Score: {score:.4f}")
                print("─" * 50)
            
            print(f"\nTotal Compatibility Score: {total_score:.4f}")
        else:
            print("No valid assignment found.")
            
    except FileNotFoundError:
        print("Error: Could not find job_seekers_with_ids.csv file.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == "__main__":
    main()

Goal reached!

=== Optimal Assignments ===

Job: JOB019
  Employee: 10241
  Match Score: 0.2741
──────────────────────────────────────────────────

Job: JOB009
  Employee: 17945
  Match Score: 0.3625
──────────────────────────────────────────────────

Job: JOB010
  Employee: 15931
  Match Score: 0.2236
──────────────────────────────────────────────────

Job: JOB011
  Employee: 14346
  Match Score: 0.1156
──────────────────────────────────────────────────

Job: JOB015
  Employee: 17072
  Match Score: 0.2217
──────────────────────────────────────────────────

Job: JOB012
  Employee: 14968
  Match Score: 0.1674
──────────────────────────────────────────────────

Job: JOB013
  Employee: 17555
  Match Score: 0.1168
──────────────────────────────────────────────────

Job: JOB014
  Employee: 17900
  Match Score: 0.2217
──────────────────────────────────────────────────

Job: JOB016
  Employee: 14696
  Match Score: 0.0206
──────────────────────────────────────────────────

Job: JOB006
  Employ